In [ ]:
!pip install ultralytics

In [ ]:
import numpy as np
from PIL import Image
import cv2
import tensorflow as tf
from keras.models import load_model
import base64
from ultralytics import YOLO

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
def f1_score_metric(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(tf.math.round(y_pred), tf.float32)
    tp = tf.reduce_sum(y_true * y_pred)
    fp = tf.reduce_sum(y_pred) - tp
    fn = tf.reduce_sum(y_true) - tp
    precision = tp / (tp + fp + tf.keras.backend.epsilon())
    recall = tp / (tp + fn + tf.keras.backend.epsilon())
    f1_score = 2 * precision * recall / (precision + recall + tf.keras.backend.epsilon())
    return f1_score

yolo = YOLO(r'/content/drive/MyDrive/Kartik /Mackerel(STACKED_MODEL)/best.pt')
cut = YOLO(r'/content/drive/MyDrive/Kartik /Mackerel(STACKED_MODEL)/cut3.pt')
eye_model = YOLO(r'/content/drive/MyDrive/Kartik /Mackerel(STACKED_MODEL)/eye.pt')
# mackerel = tf.keras.models.load_model(r'/content/drive/MyDrive/Kartik /Mackerel(STACKED_MODEL)/Mackerel2.h5')
# new_mackerel = tf.keras.models.load_model(r'/content/drive/MyDrive/Kartik /Mackerel(STACKED_MODEL)/Mackerel_softness.h5')
mackerel2 = tf.keras.models.load_model(r'/content/drive/MyDrive/Kartik /Mackerel(STACKED_MODEL)/mackerel2.h5', custom_objects={"f1_score_metric":f1_score_metric})
old_mackerel = tf.keras.models.load_model(r'/content/drive/MyDrive/Kartik /Mackerel(STACKED_MODEL)/Current_mackerel_model/mackerel(current).h5', custom_objects={"f1_score_metric":f1_score_metric})



import tensorflow as tf
from tensorflow.keras import backend as K
def dice_loss(y_true, y_pred):
    smooth = 1e-12
    intersection = K.sum(K.abs(y_true * y_pred), axis=-1)
    denominator = K.sum(K.abs(y_true) + K.abs(y_pred), axis=-1)
    dice = (2.0 * intersection + smooth) / (denominator + smooth)
    return 1 - dice

def f1_loss(y_true, y_pred):
    epsilon = 1e-7
    y_pred = K.clip(y_pred, epsilon, 1 - epsilon)
    ce_loss = categorical_crossentropy(y_true, y_pred)
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)), axis=0)
    predicted_positives = K.sum(K.round(K.clip(y_pred, 0, 1)), axis=0)
    actual_positives = K.sum(K.round(K.clip(y_true, 0, 1)), axis=0)
    precision = true_positives / (predicted_positives + K.epsilon())
    recall = true_positives / (actual_positives + K.epsilon())
    f1 = 2 * (precision * recall) / (precision + recall + K.epsilon())
    f1_loss = -K.mean(f1)
    alpha = 0.4
    combined_loss = alpha * ce_loss + (1 - alpha) * f1_loss

    return combined_loss


import tensorflow as tf
from tensorflow.keras import backend as K

def f1_metric(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    predicted_positives = K.sum(K.round(K.clip(y_pred, 0, 1)))
    possible_positives = K.sum(K.round(K.clip(y_true, 0, 1)))

    precision = true_positives / (predicted_positives + K.epsilon())
    recall = true_positives / (possible_positives + K.epsilon())

    f1 = 2 * (precision * recall) / (precision + recall + K.epsilon())
    return f1


model70 = tf.keras.models.load_model('/content/drive/MyDrive/Kartik /Mackerel(STACKED_MODEL)/Mackerel_70.h5',custom_objects={'f1_metric':f1_metric, 'dice_loss':dice_loss})
model80 = tf.keras.models.load_model('/content/drive/MyDrive/Kartik /Mackerel(STACKED_MODEL)/Mackerel80.h5',custom_objects={'f1_metric':f1_metric, 'f1_loss':f1_loss})

In [ ]:
def process_image2(image, size=(640, 640)): # for eye model
    h, w = image.shape[:2]
    aspect_ratio = w / h
    if aspect_ratio > 1:
        new_w = size[0]
        new_h = int(new_w / aspect_ratio)
    else:
        new_h = size[1]
        new_w = int(new_h * aspect_ratio)

    resized_image = cv2.resize(image, (new_w, new_h))

    pad_h = (size[1] - new_h) // 2
    pad_w = (size[0] - new_w) // 2
    padded_image = cv2.copyMakeBorder(
        resized_image, pad_h, pad_h, pad_w, pad_w, cv2.BORDER_CONSTANT, value=(255, 255, 255)
    )
    padded_image = cv2.resize(padded_image, (640,640))
    return np.asarray(padded_image)

def is_turbid_square(img): #pass grayscale image
    height, width = img.shape
    center_x, center_y = width // 2, height // 2
    square_size = 100
    top_left_x = center_x - square_size // 2
    top_left_y = center_y - square_size // 2
    bottom_right_x = top_left_x + square_size
    bottom_right_y = top_left_y + square_size
    square_img = img[top_left_y:bottom_right_y, top_left_x:bottom_right_x]
    normalized_square = square_img.astype(float) / 255.0
    white_pixels = normalized_square > 0.25
    num_white_pixels = white_pixels.sum()
    num_black_pixels = white_pixels.size - num_white_pixels
    white_black_ratio = num_white_pixels / num_black_pixels if num_black_pixels != 0 else num_white_pixels
    return white_black_ratio >= 1

def is_turbid(image):
    res = eye_model.predict(image, conf=0.5, verbose=False)
    print(len(res[0].boxes.xyxy))
    if len(res[0].boxes.xyxy) == 0:
        return None
    else:
        boxes = res[0].boxes.xyxy
        xmin, ymin, xmax, ymax = list(map(int, boxes[0]))
        eye = image[int(ymin):int(ymax), int(xmin):int(xmax)]
        eye = process_image2(eye)
        eye = cv2.cvtColor(eye, cv2.COLOR_BGR2GRAY)
        if is_turbid_square(eye):
            return 1
        else:
            return 0

def is_cut(image):
    res = cut.predict(image, conf=0.5, verbose=False)
    if len(res[0].boxes.xyxy) == 0:
        return False
    else:
        return True

def process_image(image):
    resized_image = resize_image(image, size=(224, 224))
    resized_image = cv2.resize(resized_image,(224,224))
    processed_image = resized_image/255.0
    processed_image = np.expand_dims(processed_image, axis=0)
    return tf.convert_to_tensor(processed_image, dtype=tf.float32)
def resize_image(image, size):
    h, w = image.shape[0],image.shape[1]
    aspect_ratio = w / h
    if aspect_ratio > 1:
        new_w = size[0]
        new_h = int(new_w / aspect_ratio)
    else:
        new_h = size[1]
        new_w = int(new_h * aspect_ratio)
    resized_image = cv2.resize(image, (new_w, new_h))
    padded_image = pad_image(resized_image, size)
    return padded_image
def pad_image(image, size):
    h, w = image.shape[0], image.shape[1]
    pad_h = (size[1] - h) // 2
    pad_w = (size[0] - w) // 2
    padded_image = cv2.copyMakeBorder(
        image, pad_h, pad_h, pad_w, pad_w, cv2.BORDER_CONSTANT, value=(255, 255, 255)
    )
    return padded_image
def load_image(image):
    image = np.array(image)
    processed_image = process_image(image)
    return processed_image
def threshold(predictions):
    return np.where(predictions > 0.55, 1, 0)


def freshness_prediction(image):
    decision = 0
    damage = is_cut(process_image2(image))
    if damage:
        return 'Bad'
    turbidity = is_turbid(process_image2(image))
    if turbidity==1:
        return 'Bad'
    elif turbidity==0:
        return 'Good'
    image = load_image(image)
    pred = mackerel.predict(image)
    result = np.argmax(pred)
    if result == 1:
        decision+=0.5
    else:
        decision-=0.5
    if turbidity==1:
        decision -= 0.8
    elif turbidity==0:
        decision += 0.8
    else:
        decision += 0
    if damage:
        decision -= 1.5
    if decision>0:
        return 'Good'
    elif decision==0:
        return 'Ok'
    else:
        return 'Bad'

def extract_single_image_segment(img, mask_segment, shape):
    height, width = shape[0], shape[1]
    segment = np.array(mask_segment, dtype=np.int32)
    segment = segment.reshape((-1, 2))
    mask = np.zeros((height, width), dtype=np.uint8)
    cv2.fillPoly(mask, [segment], 255)
    masked_img = cv2.bitwise_and(img, img, mask=mask)
    x, y, w, h = cv2.boundingRect(segment)
    segment_image = np.zeros((h, w, 3), dtype=np.uint8)
    segment_image[0:h, 0:w] = masked_img[y:y+h, x:x+w]
    black_mask = np.all(segment_image == [0, 0, 0], axis=-1)
    segment_image[black_mask] = [255, 255, 255]
    return segment_image

def extract_single_image_segment(img, mask_segment, shape, background='white'):
    height, width = shape[0], shape[1]
    segment = np.array(mask_segment, dtype=np.int32)
    segment = segment.reshape((-1, 2))
    mask = np.zeros((height, width), dtype=np.uint8)
    cv2.fillPoly(mask, [segment], 255)
    masked_img = cv2.bitwise_and(img, img, mask=mask)
    x, y, w, h = cv2.boundingRect(segment)
    segment_image = np.zeros((h, w, 3), dtype=np.uint8)
    segment_image[0:h, 0:w] = masked_img[y:y+h, x:x+w]

    if background.lower() == 'black':
        black_mask = np.all(segment_image == [0, 0, 0], axis=-1)
        segment_image[black_mask] = [0, 0, 0]  # Set background to black
    elif background.lower() == 'white':
        black_mask = np.all(segment_image == [0, 0, 0], axis=-1)
        segment_image[black_mask] = [255, 255, 255]  # Set background to white
    else:
        raise ValueError("Invalid background color. Use 'black' or 'white'.")

    return segment_image

def preprocess_image_with_final_mean(image, final_mean=[219.76306568173428, 218.50215572363513, 217.7583217995195], target_shape=(256,256)):
    # print('Correct')
    image = np.array(image)
    image = cv2.resize(image, target_shape)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = image.astype(np.float32)
    image -= np.array(final_mean, dtype=np.float32)
    return image/255.0

def freshness_prediction2(image):
    image = preprocess_image_with_final_mean(image)
    image = np.expand_dims(image, axis=0)
    out = ['Bad', 'Good']
    pred = mackerel2.predict(image, verbose=0)
    pred = np.argmax(pred)
    return out[pred]

def freshness_prediction_old(image, raw=False):
    image = load_image(image)
    pred = mackerel2.predict(image, verbose=0)
    if raw:
      return pred
    label = ['Bad', 'Good']
    return label[np.argmax(pred[0])]

def damage_model(image, raw=False):
    damage = is_cut(process_image2(image))
    if damage:
      if raw:
        return 0
      return 'Bad'
    else:
      if raw:
        return 1
      return 'Good'

def mackerel_with_damage(image, raw=False):
    damage = is_cut(process_image2(image))
    if damage:
      return 'bad'
    else:
      image = load_image(image)
      pred = model70.predict(image, verbose=0)
      label = ['Bad', 'Good']
      return label[np.argmax(pred[0])]


In [ ]:
def final_prediction2(image, mask=False, raw=False):
  image = cv2.imread(image)
  image = np.asarray(image)
  shape = image.shape
  try:
     pred = yolo.predict(image, conf=0.75, verbose=False)
  except:
     return None

  if len(pred[0].boxes.xyxy) == 0 or pred[0].masks == None:
     return None
  boxes = pred[0].boxes.xyxy
  masks=pred[0].masks.xy
  filtered_boxes = sorted(boxes, key=lambda bbox: bbox[0])
  sorted_masks = [masks[i] for i in sorted(range(len(masks)), key=lambda x: boxes[x][0])]
  freshness = []
  if mask:
    for mask in sorted_masks:
      extracted_image = extract_single_image_segment(image, mask, shape)
      freshness.append(mackerel_with_damage(extracted_image,raw=raw))
      break
    return freshness
  else:
    for box in filtered_boxes:
      xmin, ymin, xmax, ymax = list(map(int,box))
      extracted_image = image[ymin:ymax, xmin:xmax]
      extracted_image = resize_image(extracted_image, (640,640))
      freshness.append(mackerel_with_damage(extracted_image, raw=raw))
    return freshness

In [ ]:
import matplotlib.pyplot as plt
image = cv2.imread('/content/20230523093930124_sardine_bad.jpeg')
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
image = np.asarray(image)
shape = image.shape
pred = yolo.predict(image, conf=0.75, verbose=False)
boxes = pred[0].boxes.xyxy
masks=pred[0].masks.xy
filtered_boxes = sorted(boxes, key=lambda bbox: bbox[0])
sorted_masks = [masks[i] for i in sorted(range(len(masks)), key=lambda x: boxes[x][0])]
mask = sorted_masks[0]
extracted_image = extract_single_image_segment(image, mask, shape)
plt.imshow(extracted_image)

In [ ]:
final_prediction2('/content/drive/MyDrive/Sowmya /qZense Dataset/Final Data/Mackerel/Bad/20230520105846433_mackerel_bad.jpeg', mask=False, raw=False)

In [ ]:
from glob import glob
from tqdm import tqdm
import random
bad_images = glob("/content/drive/MyDrive/Sowmya /qZense Dataset/Final Data/Mackerel/Bad/*")
good_images = glob('/content/drive/MyDrive/Sowmya /qZense Dataset/Final Data/Mackerel/Good/*')
good_images = random.sample(good_images, 300)
bad_images = random.sample(bad_images, 300)

In [ ]:
len(bad_images),len(good_images)

In [ ]:
results= {}
actual = {}

actual_labels = []
predicted_labels = []
for img in tqdm(bad_images):
  try:
    lst = final_prediction2(img, mask=True)
    if lst == None:
      continue
    actual_labels += ['Bad'] * len(lst)
    predicted_labels+=lst
    good = lst.count('Good')
    bad = lst.count('Bad')
    actual['Bad'] = results.get('Bad', 0) + bad + good
    results['Good'] = results.get('Good', 0) + good
    results['Bad'] = results.get('Bad', 0) + bad
  except:
    continue


for img in tqdm(good_images):
  if len(actual_labels) >= len(bad_images) * 2:
    break
  try:
    lst = final_prediction2(img, mask=True)
    if lst == None:
      continue
    actual_labels += ['Good'] * len(lst)
    predicted_labels+=lst
    good = lst.count('Good')
    bad = lst.count('Bad')
    actual['Good'] = results.get('Good', 0) + bad + good
    results['Good'] = results.get('Good', 0) + good
    results['Bad'] = results.get('Bad', 0) + bad
  except:
    continue

In [ ]:
# results= {}
# actual = {}
# raw_labels=[]
# actual_labels = []
# predicted_labels = []
# for img in tqdm(bad_images):
#   try:
#       lst = final_prediction2(img)
#       raw_labels+=final_prediction2(img, raw=True)
#   except:
#       continue
#   if lst == None:
#     continue
#   actual_labels += [0] * len(lst)
#   predicted_labels+=lst
#   good = lst.count(1)
#   bad = lst.count(0)
#   actual['Bad'] = results.get(0, 0) + bad + good
#   results['Good'] = results.get(1, 0) + good
#   results['Bad'] = results.get(0, 0) + bad


# for img in tqdm(good_images):
#   try:
#       lst = final_prediction2(img)
#       raw_labels+=final_prediction2(img, raw=True)
#   except:
#       continue
#   if lst == None:
#     continue
#   actual_labels += [1] * len(lst)
#   predicted_labels+=lst
#   good = lst.count(1)
#   bad = lst.count(0)
#   actual['Good'] = results.get(1, 0) + bad + good
#   results['Good'] = results.get(1, 0) + good
#   results['Bad'] = results.get(0, 0) + bad

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, accuracy_score

In [ ]:
cm = confusion_matrix(actual_labels, predicted_labels, labels=['Bad', 'Good'])
plt.figure(figsize=(6, 4))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.colorbar()
tick_marks = np.arange(len(['Bad', 'Good']))
plt.xticks(tick_marks, ['Bad', 'Good'], rotation=0)
plt.yticks(tick_marks, ['Bad', 'Good'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
for i in range(len(['Bad', 'Good'])):
    for j in range(len(['Bad', 'Good'])):
        plt.text(j, i, format(cm[i, j], 'd'), ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.tight_layout()
accuracy = accuracy_score(actual_labels, predicted_labels)
print(f"Overall Accuracy: {accuracy:.2%}")
plt.show()


#'''USE FOR DAMAGE MODEL'''
# cm = confusion_matrix(actual_labels, predicted_labels, labels=['Bad', 'Good'])
# plt.figure(figsize=(6, 4))
# plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
# plt.title('Confusion Matrix')
# plt.colorbar()
# tick_marks = np.arange(len(['Bad', 'Good']))
# plt.xticks(tick_marks, ['Damage_Present', 'Damage_Absent'], rotation=0)
# plt.yticks(tick_marks, ['Damage_Present', 'Damage_Absent'])
# plt.xlabel('Predicted')
# plt.ylabel('Actual')
# for i in range(len(['Bad', 'Good'])):
#     for j in range(len(['Bad', 'Good'])):
#         plt.text(j, i, format(cm[i, j], 'd'), ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
# plt.tight_layout()
# accuracy = accuracy_score(actual_labels, predicted_labels)
# print(f"Overall Accuracy: {accuracy:.2%}")
# plt.show()

In [ ]:
# from sklearn.metrics import roc_curve, roc_auc_score, f1_score
# import matplotlib.pyplot as plt
# model_predictions = raw_labels
# true_labels = actual_labels

# # Compute ROC curve
# fpr, tpr, thresholds = roc_curve(true_labels, model_predictions)

# # Plot ROC curve
# plt.figure(figsize=(8, 6))
# plt.plot(fpr, tpr, label='ROC curve')
# plt.plot([0, 1], [0, 1], 'k--', label='Random Guessing')
# plt.xlabel('False Positive Rate')
# plt.ylabel('True Positive Rate')
# plt.title('Receiver Operating Characteristic (ROC) Curve')
# plt.legend()

# # Calculate AUC (Area Under the ROC Curve)
# roc_auc = roc_auc_score(true_labels, model_predictions)
# print(f"AUC (Area Under the ROC Curve): {roc_auc:.2f}")

# # Find the optimal threshold based on F1-score
# f1_scores = 2 * (tpr * (1 - fpr)) / (tpr + (1 - fpr))
# optimal_threshold = thresholds[np.argmax(f1_scores)]
# print(f"Optimal Threshold for F1-Score: {optimal_threshold:.2f}")

# diff = tpr - fpr
# idx = np.argmax(diff)
# optimal_threshold2 = thresholds[idx]
# print(f"Optimal Threshold 2: {optimal_threshold2:.2f}")

In [ ]:
# label_mapping = {'Bad': 'Damage_Present', 'Good': 'Damage_Absent'}
# actual_labels_mapped = [label_mapping[label] for label in actual_labels]
# predicted_labels_mapped = [label_mapping[label] for label in predicted_labels]

In [ ]:
from sklearn.metrics import classification_report
class_report = classification_report(actual_labels, predicted_labels, labels=['Bad', 'Good'])

print("Classification Report:\n")
print(class_report)

fig, ax = plt.subplots(figsize=(8, 6))
ax.axis('off')
ax.text(0.1, 0.1, class_report, va='top', bbox={'facecolor': 'wheat', 'alpha': 0.5, 'boxstyle': 'round'})

# Save the figure as an image
fig.savefig('classification_report.png', bbox_inches='tight', dpi=300)
plt.close()




# from sklearn.metrics import classification_report
# class_report = classification_report(actual_labels, predicted_labels, target_names=['Damage_Present', 'Damage_Absent'])

# print("Classification Report:\n")
# print(class_report)

# fig, ax = plt.subplots(figsize=(8, 6))
# ax.axis('off')
# ax.text(0.1, 0.1, class_report, va='top', bbox={'facecolor': 'wheat', 'alpha': 0.5, 'boxstyle': 'round'})

# # Save the figure as an image
# fig.savefig('classification_report.png', bbox_inches='tight', dpi=300)
# plt.close()

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

def process_image(image):
    resized_image = resize_image(image, size=(224, 224))
    resized_image = cv2.resize(resized_image,(224,224))
#     equalized_image = equalize_contrast(resized_image)
    processed_image = resized_image / 255.0
    return tf.convert_to_tensor(processed_image, dtype=tf.float32)

def resize_image(image, size):
    h, w = image.shape[:2]
    aspect_ratio = w / h
    if aspect_ratio > 1:
        new_w = size[0]
        new_h = int(new_w / aspect_ratio)
    else:
        new_h = size[1]
        new_w = int(new_h * aspect_ratio)
    resized_image = cv2.resize(image, (new_w, new_h))
    padded_image = pad_image(resized_image, size)
    return padded_image

def pad_image(image, size):
    h, w = image.shape[0],image.shape[1]
    pad_h = (size[1] - h) // 2
    pad_w = (size[0] - w) // 2
    padded_image = cv2.copyMakeBorder(
        image, pad_h, pad_h, pad_w, pad_w, cv2.BORDER_CONSTANT, value=(255, 255, 255)
    )
    return padded_image

def equalize_contrast(image):
    lab_image = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    lab_planes = list(cv2.split(lab_image))  # Convert tuple to list
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    lab_planes[0] = clahe.apply(lab_planes[0])
    equalized_image = cv2.merge(lab_planes)
    equalized_image = cv2.cvtColor(equalized_image, cv2.COLOR_LAB2BGR)
    return equalized_image

def load_image(image_path):
    image = cv2.imread(image_path)
    processed_image = process_image(image)
    return processed_image


test_datagen = ImageDataGenerator(
    preprocessing_function=process_image)

batch_size = 64
target_size = (224, 224)

test = test_datagen.flow_from_directory(
    directory = "/content/drive/MyDrive/Sowmya /qZense Dataset/Final Data/Mackerel",
    target_size=target_size,
    batch_size=batch_size,
    shuffle=False,
    class_mode='categorical',
)


import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import itertools

def plot_confusion_matrix(cm, classes,
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues):
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="black" if cm[i, j] > thresh else "red")

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.show()

def evaluate_binary_classification_model(model, test_generator):
    y_true = test_generator.classes
    y_pred = model.predict(test_generator, verbose=1)
    y_pred_labels = np.argmax(y_pred, axis=1)

    print("Classification Report:\n")
    print(classification_report(y_true, y_pred_labels, target_names=test_generator.class_indices.keys()))

    # Plot confusion matrix
    cnf_matrix = confusion_matrix(y_true, y_pred_labels)
    plot_confusion_matrix(cnf_matrix, classes=test_generator.class_indices.keys())



In [ ]:
evaluate_binary_classification_model(model70,test)

In [ ]:
evaluate_binary_classification_model(model80,test)

In [ ]:
evaluate_binary_classification_model(mackerel2,test)